# 06 · การประมาณค่าฝนเชิงพื้นที่: IDW และ Thiessen Polygon
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apichayaice7835-png/Teach-Python-Geography/blob/main/06_Rainfall_Interpolation_IDW_Thiessen.ipynb)

**เวลา:** 3 ชั่วโมง | **ระดับ:** 🔴 ยาก (ต้องผ่าน Notebook 02–05 มาก่อน)

**ปัญหา:** เรามีค่าฝนเฉพาะที่ *จุด* สถานี แต่ต้องการรู้ฝนทั้ง *พื้นที่* (เช่น ลุ่มน้ำ จังหวัด)

### 1) Thiessen (Voronoi) Polygon — วิธีมาตรฐานในงานอุทกวิทยา
ทุกจุดในพื้นที่ใช้ค่าของสถานีที่ **ใกล้ที่สุด** ฝนเฉลี่ยพื้นที่ = ค่าเฉลี่ยถ่วงน้ำหนักด้วยพื้นที่
$$\bar{P} = \frac{\sum A_i P_i}{\sum A_i}$$

### 2) Inverse Distance Weighting (IDW)
ค่าที่จุดใด ๆ = ค่าเฉลี่ยถ่วงน้ำหนักด้วย *ส่วนกลับของระยะทางยกกำลัง p*
$$\hat{Z}(x_0) = \frac{\sum_{i=1}^{n} w_i Z_i}{\sum_{i=1}^{n} w_i}, \quad w_i = \frac{1}{d_i^{\,p}}$$
p มาก → สถานีใกล้มีอิทธิพลมาก (ผิวเป็นหย่อม ๆ), p น้อย → ผิวเรียบ

> ⚠️ **ข้อจำกัด:** ทั้งสองวิธีไม่ใช้ข้อมูลภูมิประเทศ (ความสูง ทิศด้านรับลม) และให้ค่าไม่เกินช่วงค่าที่สถานีวัดได้ — ในพื้นที่ภูเขาฝนจริงอาจสูงกว่ามาก

In [ ]:
# ============================================================
# ตั้งค่าฟอนต์ภาษาไทยให้ Matplotlib (ถ้าไม่ตั้ง ตัวอักษรไทยจะเป็นสี่เหลี่ยม □□□)
# ============================================================
import glob, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

def setup_thai_font():
    fonts = glob.glob("/usr/share/fonts/**/Loma*.[ot]tf", recursive=True)
    if not fonts:   # บน Colab ต้องติดตั้งก่อน (ใช้เวลา ~10 วินาที ครั้งเดียวต่อ session)
        subprocess.run("apt-get -qq install -y fonts-thai-tlwg > /dev/null", shell=True)
        fonts = glob.glob("/usr/share/fonts/**/Loma*.[ot]tf", recursive=True)
    for f in fonts:
        fm.fontManager.addfont(f)
    if fonts:
        plt.rcParams["font.family"] = "Loma"
        print("✅ ใช้ฟอนต์ไทย Loma")
    else:
        print("⚠️ ติดตั้งฟอนต์ไทยไม่ได้ ใช้ฟอนต์เริ่มต้นแทน")
    plt.rcParams["axes.unicode_minus"] = False   # ให้เครื่องหมายลบแสดงถูกต้อง

setup_thai_font()

In [ ]:
# ============================================================
# เซลล์โหลดข้อมูล (ใช้ซ้ำทุก Notebook)
# ลำดับการหา: 1) โฟลเดอร์ data/ ในเครื่อง  2) GitHub ของอาจารย์
# ============================================================
import os
import pandas as pd

GITHUB_USER = "apichayaice7835-png"      # ← ชื่อบัญชี GitHub ของอาจารย์
REPO        = "Teach-Python-Geography"   # ← ชื่อ repository
BASE_URL    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main/data/"

def data_path(filename):
    """คืน path ของไฟล์: ถ้ามีในเครื่องใช้ในเครื่อง ถ้าไม่มีใช้ลิงก์ GitHub"""
    for folder in ["data", f"{REPO}/data", "/content/data"]:
        p = os.path.join(folder, filename)
        if os.path.exists(p):
            return p
    return BASE_URL + filename

print("ไฟล์สถานี :", data_path("rain_stations_north.csv"))
print("ไฟล์ฝนรายเดือน:", data_path("rain_monthly_2020_2024.csv"))

In [ ]:
import numpy as np
import geopandas as gpd
import shapely
from shapely.geometry import MultiPoint

# ----- ข้อมูลสถานี + ฝนรายปีเฉลี่ย -----
stations = pd.read_csv(data_path("rain_stations_north.csv"))
rain = pd.read_csv(data_path("rain_monthly_2020_2024.csv"), parse_dates=["date"])
rain["year"] = rain["date"].dt.year
annual = (rain.groupby(["station_id", "year"])["rain_mm"].sum()
              .groupby("station_id").mean().rename("annual_mm").reset_index())
st = gpd.GeoDataFrame(stations.merge(annual, on="station_id"),
                      geometry=gpd.points_from_xy(stations["lon"], stations["lat"]),
                      crs=4326).to_crs(32647)          # ทำงานใน UTM (เมตร)

# ----- ขอบเขต -----
PROV_URL = "https://raw.githubusercontent.com/apisit/thailand.json/master/thailand.json"
local = data_path("thailand_provinces.geojson")
prov = gpd.read_file(local if os.path.exists(local) else PROV_URL)
NORTH = ["Chiang Mai","Chiang Rai","Lampang","Lamphun","Mae Hong Son","Nan","Phayao","Phrae","Uttaradit"]
north = prov[prov["name"].isin(NORTH)].rename(columns={"name": "prov_name"}).to_crs(32647)
region = north.union_all()                            # พื้นที่ภาคเหนือรวมเป็น polygon เดียว
print(len(st), "สถานี |", len(north), "จังหวัด | พื้นที่รวม %.0f ตร.กม." % (region.area / 1e6))

## 1. Thiessen Polygon

In [ ]:
# สร้าง Voronoi จากจุดสถานี แล้วตัด (clip) ด้วยขอบเขตภาคเหนือ
vor = shapely.voronoi_polygons(MultiPoint(list(st.geometry)), extend_to=region)
thiessen = gpd.GeoDataFrame(geometry=list(vor.geoms), crs=32647)
thiessen = gpd.sjoin(thiessen, st[["station_id", "name_th", "annual_mm", "geometry"]],
                     predicate="contains").drop(columns="index_right")   # ผูกแต่ละ polygon กับสถานีข้างใน
thiessen = gpd.clip(thiessen, region)
thiessen["area_km2"] = thiessen.area / 1e6
thiessen["weight"] = thiessen["area_km2"] / thiessen["area_km2"].sum()

print("ฝนเฉลี่ยเลขคณิต  = %.0f มม." % st["annual_mm"].mean())
print("ฝนเฉลี่ย Thiessen = %.0f มม." % (thiessen["weight"] * thiessen["annual_mm"]).sum())
thiessen[["name_th", "area_km2", "weight", "annual_mm"]].sort_values("weight", ascending=False).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
thiessen.plot(ax=ax, column="annual_mm", cmap="YlGnBu", edgecolor="white", legend=True,
              legend_kwds={"label": "ฝนรายปี (มม.)", "shrink": 0.6})
north.boundary.plot(ax=ax, color="black", lw=0.6)
st.plot(ax=ax, color="red", marker="^", markersize=30)
for _, r in st.iterrows():
    ax.annotate(r["name_th"], (r.geometry.x, r.geometry.y), fontsize=7, xytext=(3, 3), textcoords="offset points")
ax.set_title("Thiessen Polygon ของสถานีวัดน้ำฝน"); ax.set_axis_off(); plt.show()

## 2. IDW ด้วย NumPy (เขียนเอง เพื่อเข้าใจสูตร)

In [ ]:
def idw(xs, ys, zs, xi, yi, power=2):
    """
    xs, ys, zs : พิกัดและค่าของสถานี (array 1 มิติ ยาว n)
    xi, yi     : พิกัดจุดที่ต้องการประมาณค่า (array 1 มิติ ยาว m)
    คืนค่า     : array ยาว m
    """
    # ระยะทางจากทุกจุดเป้าหมาย (m) ไปทุกสถานี (n) → เมทริกซ์ m × n ด้วย broadcasting
    d = np.sqrt((xi[:, None] - xs[None, :])**2 + (yi[:, None] - ys[None, :])**2)
    d = np.maximum(d, 1e-6)                 # กันหารด้วยศูนย์เมื่อจุดตรงกับสถานีพอดี
    w = 1.0 / d**power
    return (w * zs).sum(axis=1) / w.sum(axis=1)

xs, ys, zs = st.geometry.x.values, st.geometry.y.values, st["annual_mm"].values

# ทดสอบกับจุดเดียว: กึ่งกลางระหว่างเชียงใหม่กับลำปาง
test = idw(xs, ys, zs, np.array([(xs[0] + xs[7]) / 2]), np.array([(ys[0] + ys[7]) / 2]))
print(f"เชียงใหม่ {zs[0]:.0f} | ลำปาง {zs[7]:.0f} | จุดกึ่งกลาง (IDW) ≈ {test[0]:.0f} มม.")

In [ ]:
# สร้างกริดความละเอียด 2 กม. ครอบคลุมภาคเหนือ
RES = 2000
minx, miny, maxx, maxy = region.bounds
gx = np.arange(minx, maxx, RES) + RES / 2
gy = np.arange(miny, maxy, RES) + RES / 2
GX, GY = np.meshgrid(gx, gy)

Z = idw(xs, ys, zs, GX.ravel(), GY.ravel(), power=2).reshape(GX.shape)   # ravel = ทำเป็น 1 มิติ
inside = shapely.contains_xy(region, GX, GY)                              # mask ด้วยขอบเขต
Z_masked = np.where(inside, Z, np.nan)
print("กริด", Z.shape, "| จำนวนช่องในพื้นที่", inside.sum())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 9))
levels = np.arange(900, 1751, 50)
cf = ax.contourf(GX, GY, Z_masked, levels=levels, cmap="YlGnBu", extend="both")
cs = ax.contour(GX, GY, Z_masked, levels=levels[::2], colors="k", linewidths=0.4)
ax.clabel(cs, fmt="%d", fontsize=7)                       # ตัวเลขบนเส้นชั้นน้ำฝน (isohyet)
north.boundary.plot(ax=ax, color="dimgray", lw=0.8)
st.plot(ax=ax, color="red", marker="^", markersize=30)
fig.colorbar(cf, ax=ax, shrink=0.6, label="ฝนรายปี (มม.)")
ax.set_title("เส้นชั้นน้ำฝน (Isohyet) จาก IDW, p = 2"); ax.set_axis_off()
fig.savefig("idw_isohyet.png", dpi=200, bbox_inches="tight"); plt.show()

## 3. เลือกค่า power ที่ดีที่สุดด้วย Leave-One-Out Cross-Validation (LOOCV)
เอาสถานีออกทีละแห่ง → ประมาณค่าจากสถานีที่เหลือ → เทียบกับค่าจริง
$$RMSE = \sqrt{\frac{1}{n}\sum (\hat{Z}_i - Z_i)^2}$$

In [ ]:
def loocv(power):
    pred = np.empty_like(zs)
    for i in range(len(zs)):
        keep = np.arange(len(zs)) != i                    # ตัดสถานี i ออก
        pred[i] = idw(xs[keep], ys[keep], zs[keep], xs[[i]], ys[[i]], power)[0]
    return pred

results = []
for p in [0.5, 1, 1.5, 2, 3, 4]:
    pred = loocv(p)
    results.append({"power": p,
                    "RMSE": np.sqrt(np.mean((pred - zs)**2)),
                    "MAE":  np.mean(np.abs(pred - zs)),
                    "Bias": np.mean(pred - zs)})
cv = pd.DataFrame(results).round(1)
best_p = cv.loc[cv["RMSE"].idxmin(), "power"]
print("power ที่ให้ RMSE ต่ำสุด =", best_p)
if best_p == cv["power"].max():
    print("⚠️ ค่าต่ำสุดอยู่ที่ขอบของช่วงที่ทดสอบ → ลองขยายช่วง power และสังเกตว่า RMSE ต่างกันมากหรือไม่")
cv

In [ ]:
pred = loocv(best_p)
fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
ax[0].plot(cv["power"], cv["RMSE"], "o-"); ax[0].set_xlabel("power (p)"); ax[0].set_ylabel("RMSE (มม.)")
ax[0].set_title("RMSE จาก LOOCV")
ax[1].scatter(zs, pred, edgecolor="k")
lim = [zs.min() - 100, zs.max() + 100]; ax[1].plot(lim, lim, "r--", label="1:1")
for n, a, b in zip(st["name_th"], zs, pred):
    ax[1].annotate(n, (a, b), fontsize=7)
ax[1].set_xlabel("ค่าจริง (มม.)"); ax[1].set_ylabel("ค่าประมาณ (มม.)"); ax[1].legend()
ax[1].set_title(f"ค่าจริง vs ค่าประมาณ (p = {best_p})"); plt.tight_layout(); plt.show()

## 4. ฝนเฉลี่ยระดับจังหวัด: เปรียบเทียบ 3 วิธี

In [ ]:
# (ก) เฉลี่ยเลขคณิตของสถานีในจังหวัด
arith = gpd.sjoin(st, north[["prov_name", "geometry"]], predicate="within").groupby("prov_name")["annual_mm"].mean()

# (ข) Thiessen: ซ้อนทับ (overlay) Thiessen กับจังหวัด แล้วถ่วงน้ำหนักด้วยพื้นที่
ov = gpd.overlay(thiessen[["annual_mm", "geometry"]], north[["prov_name", "geometry"]], how="intersection")
ov["a"] = ov.area
thies = ov.groupby("prov_name").apply(lambda g: np.average(g["annual_mm"], weights=g["a"]), include_groups=False)

# (ค) IDW: ค่าเฉลี่ยของทุกช่องกริดที่อยู่ในจังหวัด
Z_best = idw(xs, ys, zs, GX.ravel(), GY.ravel(), best_p).reshape(GX.shape)
idw_mean = {r["prov_name"]: np.nanmean(np.where(shapely.contains_xy(r.geometry, GX, GY), Z_best, np.nan))
            for _, r in north.iterrows()}

comp = pd.DataFrame({"เลขคณิต": arith, "Thiessen": thies, "IDW": pd.Series(idw_mean)}).round(0)
comp["n_สถานี"] = gpd.sjoin(st, north[["prov_name", "geometry"]], predicate="within").groupby("prov_name").size()
comp

In [ ]:
comp[["เลขคณิต", "Thiessen", "IDW"]].plot.bar(figsize=(10, 4.5), rot=30)
plt.ylabel("ฝนรายปีเฉลี่ย (มม.)"); plt.title("ฝนเฉลี่ยระดับจังหวัด: เปรียบเทียบ 3 วิธี")
plt.ylim(800, 1800); plt.tight_layout(); plt.show()

### 🧠 คำถามอภิปราย
1. จังหวัดที่มีสถานีเพียง 1 แห่ง (เช่น ลำพูน พะเยา) ค่าจาก 3 วิธีต่างกันมากหรือน้อย เพราะอะไร
2. สถานีดอยอ่างขาง (สูง ~1,400 ม.) มีอิทธิพลต่อ Thiessen polygon ของเชียงใหม่อย่างไร เหมาะสมหรือไม่
3. ถ้าต้องการรวมผลของภูมิประเทศ ควรใช้วิธีใด (เช่น regression กับความสูง + residual kriging)

---
## ✏️ แบบฝึกหัด
**🟡** 1) ทำ IDW ของฝน *เดือนสิงหาคม* แทนฝนรายปี  2) เปลี่ยนความละเอียดกริดเป็น 5 กม. และ 1 กม. เปรียบเทียบเวลาและผลลัพธ์

**🔴** 3) ใช้ `scipy.interpolate.griddata` (method="linear", "cubic") เปรียบเทียบกับ IDW ด้วย LOOCV
4) สร้าง IDW แบบใช้เฉพาะ *k สถานีที่ใกล้ที่สุด* (k = 4) ด้วย `scipy.spatial.cKDTree`
5) นำเข้าขอบเขตลุ่มน้ำ (เช่น ลุ่มน้ำน่าน) จากไฟล์ของตนเอง แล้วคำนวณฝนเฉลี่ยลุ่มน้ำด้วย Thiessen